<a href="https://colab.research.google.com/github/chathumiamarasinghe/ANOVA-Test/blob/main/ODIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd

# Load CSV
df = pd.read_csv("odin-real-time-outages-county.csv")

# View first rows
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'odin-real-time-outages-county.csv'

In [ ]:
df['reported_datetime'] = pd.to_datetime(df['reportedStartTime'], errors='coerce')

# Extract useful time features
df['year'] = df['reported_datetime'].dt.year
df['month'] = df['reported_datetime'].dt.month
df['day'] = df['reported_datetime'].dt.day
df['hour'] = df['reported_datetime'].dt.hour

In [37]:
# Parse with UTC
df['reported_datetime'] = pd.to_datetime(
    df['reportedStartTime'],
    errors='coerce',
    utc=True
)

df['EstimatedRestorationTime'] = pd.to_datetime(
    df['EstimatedRestorationTime'],
    errors='coerce',
    utc=True
)

# Remove timezone (convert to naive)
df['reported_datetime'] = df['reported_datetime'].dt.tz_localize(None)
df['EstimatedRestorationTime'] = df['EstimatedRestorationTime'].dt.tz_localize(None)


/tmp/ipykernel_13819/869338123.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['EstimatedRestorationTime'] = pd.to_datetime(


In [ ]:
df[['latitude', 'longitude']] = df['geo_point_2d'].str.split(',', expand=True)

df['latitude'] = df['latitude'].astype(float)
df['longitude'] = df['longitude'].astype(float)

In [ ]:
df['metersAffected'] = pd.to_numeric(df['metersAffected'], errors='coerce')
df['customersRestored'] = pd.to_numeric(df['customersRestored'], errors='coerce')

In [ ]:
df['EstimatedRestorationTime'] = pd.to_datetime(df['EstimatedRestorationTime'], errors='coerce')

df['restoration_hours'] = (
    (df['EstimatedRestorationTime'] - df['reported_datetime'])
    .dt.total_seconds() / 3600
)

In [ ]:
df_clean = df[[
    'cause',
    'metersAffected',
    'reported_datetime',
    'year',
    'month',
    'hour',
    'county',
    'state',
    'latitude',
    'longitude',
    'restoration_hours'
]]

df_clean.to_csv("outages_cleaned.csv", index=False)

In [42]:
# Check missing values
print(df.isna().sum())

cause                        578
communityDescriptor            0
metersAffected                 0
reportedStartTime              0
statusKind                   653
utilityDisclaimer           1575
Incident                       0
customersRestored           1530
EstimatedRestorationTime    1629
causeKind                   1273
incident_cause               578
incident_location              0
incident_location_kind         0
name                           0
utility_id                     0
county                         0
state                          0
geom                           0
geo_point_2d                   0
centroid                       0
latitude                       0
longitude                      0
date                           0
reported_datetime              0
year                           0
month                          0
day                            0
hour                           0
restoration_hours           1629
dtype: int64


In [43]:
# Check missing values
print(df_clean.isna().sum())

cause                 578
metersAffected          0
reported_datetime       0
year                    0
month                   0
hour                    0
county                  0
state                   0
latitude                0
longitude               0
restoration_hours    1629
dtype: int64


In [44]:
# Keep only rows with datetime
df_model = df_clean.dropna(subset=['reported_datetime']).copy()

print("Rows available for modeling:", len(df_model))

Rows available for modeling: 1629


In [45]:
import requests
import time

# Function to get historical weather
def get_weather(lat, lon, date):
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={date}&end_date={date}&hourly=temperature_2m,windspeed_10m,precipitation,snowfall"

    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        # Take first hour as representative
        if "hourly" in data:
            temp = data["hourly"]["temperature_2m"][0]
            wind = data["hourly"]["windspeed_10m"][0]
            rain = data["hourly"]["precipitation"][0]
            snow = data["hourly"].get("snowfall", [0])[0]
            return temp, wind, rain, snow
        else:
            return None, None, None, None
    except:
        return None, None, None, None

# Prepare date string
df_model['date'] = pd.to_datetime(df_model['reported_datetime']).dt.strftime('%Y-%m-%d')

# Initialize lists
temps, winds, rains, snows = [], [], [], []

for i, row in df_model.iterrows():
    temp, wind, rain, snow = get_weather(row['latitude'], row['longitude'], row['date'])
    temps.append(temp)
    winds.append(wind)
    rains.append(rain)
    snows.append(snow)
    time.sleep(1)  # avoid API throttling

# Add to dataframe
df_model['temperature'] = temps
df_model['wind_speed'] = winds
df_model['precipitation'] = rains
df_model['snowfall'] = snows


KeyboardInterrupt: 

In [ ]:
print(df_model.head())

In [ ]:
# Save enriched dataset
df_model.to_csv("outages_weather_enriched.csv", index=False)

In [ ]:
df_model['weekday'] = pd.to_datetime(df_model['reported_datetime']).dt.weekday

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
import pandas as pd

In [ ]:
# Select features
features = ['year','month','hour','latitude','longitude','temperature','wind_speed','precipitation','snowfall']
X = df_model[features]

# Target
y = df_model['metersAffected']

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# XGBoost Regressor
model = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1)
model.fit(X_train, y_train)


In [21]:
# Feature importance
importance = model.feature_importances_
for f, imp in zip(features, importance):
    print(f"{f}: {imp:.3f}")

NameError: name 'model' is not defined

Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
for f, imp in zip(features, rf.feature_importances_):
    print(f"{f}: {imp:.3f}")

LightGBM Regressor

In [ ]:
import lightgbm as lgb

# Create dataset
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1
}

# Use callbacks for early stopping
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=100,
    valid_sets=[lgb_eval],
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

# Feature importance
importance = gbm.feature_importance()
for f, imp in zip(features, importance):
    print(f"{f}: {imp}")



---



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import zipfile
import os

# This creates the upload button
uploaded = files.upload()

# Get the name of the file you just uploaded
zip_filename = list(uploaded.keys())[0]

# Get the name of the file you just uploaded
filename = list(uploaded.keys())[0]

# # Unzip the data
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('my_dataset')

print(f"Successfully unzipped: {zip_filename}")
print("Files inside:", os.listdir('my_dataset'))

In [ ]:
import os

readme_folder = 'my_dataset/correlated_outage'
files_inside = os.listdir(readme_folder)
print("Files inside correlated_outage folder:", files_inside)

In [ ]:
import pandas as pd
import os

folder = 'my_dataset/correlated_outage'

# List of merged CSVs you want to load
merged_files = [
    'eaglei_outages_2014_merged.csv',
    'eaglei_outages_2015_merged.csv',
    'eaglei_outages_2016_merged.csv',
    'eaglei_outages_2017_merged.csv',
    'eaglei_outages_2018_merged.csv',
    'eaglei_outages_2019_merged.csv',
    'eaglei_outages_2020_merged.csv',
    'eaglei_outages_2021_merged.csv',
    'eaglei_outages_2022_merged.csv',
    'eaglei_outages_2023_merged.csv'
]

# Load all into one DataFrame
df_list = []
for f in merged_files:
    path = os.path.join(folder, f)
    df = pd.read_csv(path)
    df_list.append(df)

df_outages = pd.concat(df_list, ignore_index=True)
print("Shape of combined outages:", df_outages.shape)
print(df_outages.head())

In [ ]:
df_outages.to_csv("/content/drive/MyDrive/combined_outages.csv", index=False)

In [ ]:
!pip install geopy us

In [ ]:
#!pip uninstall meteostat -y
!pip install meteostat

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/combined_outages.csv')

# Get unique state-county combinations
unique_counties = df[['state', 'county']].drop_duplicates()

print("Unique counties:", unique_counties.shape)
unique_counties.head()

In [ ]:
import geopandas as gpd

url = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"
counties = gpd.read_file(url)

print(counties.columns)

In [ ]:
counties['fips'] = counties['STATEFP'] + counties['COUNTYFP']

In [ ]:
counties['longitude'] = counties.geometry.centroid.x
counties['latitude'] = counties.geometry.centroid.y

county_centroids = counties[['fips', 'latitude', 'longitude']]

county_centroids.head()

In [ ]:
import pandas as pd

#df = pd.read_csv('combined_outages.csv')

df['fips'] = df['fips'].astype(str).str.zfill(5)

df = df.merge(county_centroids, on='fips', how='left')

print(df[['fips','county','latitude','longitude']].head())

In [ ]:
df.head()

In [ ]:
# df['start_time'] = pd.to_datetime(df['start_time'])
# df['year_quarter'] = df['start_time'].dt.to_period('Q')

In [ ]:
df['start_time'] = pd.to_datetime(df['start_time'])

full_start_date = df['start_time'].min().date()
full_end_date = df['start_time'].max().date()

print(full_start_date, full_end_date)

In [1]:
!pip uninstall meteostat -y
!pip install meteostat

Found existing installation: meteostat 2.1.3
Uninstalling meteostat-2.1.3:
  Successfully uninstalled meteostat-2.1.3
  Using cached meteostat-2.1.3-py3-none-any.whl.metadata (5.2 kB)
Using cached meteostat-2.1.3-py3-none-any.whl (92 kB)


In [ ]:
# unique_county_quarters = df[['fips','latitude','longitude','year_quarter']].drop_duplicates()

# print("Unique county-quarter pairs:", unique_county_quarters.shape)

In [ ]:
unique_counties = (
    df[['fips', 'state', 'county', 'latitude', 'longitude']].drop_duplicates()
)
print("Unique counties:", unique_counties.shape)

years = range(full_start_date.year, full_end_date.year + 1)

In [ ]:
print(unique_counties.columns)
print(unique_counties.head())

In [ ]:
# # Get unique county-quarter pairs with coordinates
# unique_counties = df[['fips', 'state', 'county', 'latitude', 'longitude', 'date']].drop_duplicates()

# print(unique_counties.head())

In [ ]:
# start_date = df['start_time'].min()
# end_date = df['start_time'].max()

# print(start_date, end_date)

In [ ]:
# df['start_time'] = pd.to_datetime(df['start_time'])

# start_date = df['start_time'].min().date()
# end_date = df['start_time'].max().date()

# print(start_date, end_date)

In [ ]:
# unique_county_dates = df[['fips', 'latitude', 'longitude', 'date']].drop_duplicates()

In [ ]:
# import requests
# import pandas as pd

# def fetch_weather(lat, lon, start_date, end_date):
#     """
#     Fetch daily weather for a given latitude, longitude and date range.
#     Uses Open-Meteo API as an example.
#     """
#     try:
#         url = (
#             f"https://archive-api.open-meteo.com/v1/archive"
#             f"?latitude={lat}&longitude={lon}"
#             f"&start_date={start_date}&end_date={end_date}"
#             f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snow_depth,windspeed_10m_max"
#             f"&timezone=America/New_York"
#         )
#         response = requests.get(url, timeout=30)
#         data = response.json()

#         if "daily" in data:
#             df = pd.DataFrame(data['daily'])
#             df['date'] = pd.to_datetime(df['time'])
#             return df
#         else:
#             return None
#     except Exception as e:
#         print(f"Error fetching weather for {lat},{lon}: {e}")
#         return None

In [ ]:
# import requests
# import pandas as pd

# def fetch_weather_quarter(row):
#     lat = row.latitude
#     lon = row.longitude
#     fips = row.fips
#     quarter = row.year_quarter

#     # Convert quarter to start/end dates
#     start_date = quarter.start_time.date()
#     end_date = quarter.end_time.date()

#     try:
#         url = (
#             f"https://archive-api.open-meteo.com/v1/archive"
#             f"?latitude={lat}&longitude={lon}"
#             f"&start_date={start_date}&end_date={end_date}"
#             f"&daily=temperature_2m_max,temperature_2m_min,"
#             f"precipitation_sum,snow_depth,windspeed_10m_max"
#             f"&timezone=America/New_York"
#         )

#         response = requests.get(url, timeout=30)
#         data = response.json()

#         if "daily" in data:
#             weather_df = pd.DataFrame(data['daily'])
#             weather_df['date'] = pd.to_datetime(weather_df['time'])
#             weather_df['fips'] = fips
#             weather_df['year_quarter'] = quarter
#             return weather_df
#         else:
#             return None

#     except Exception as e:
#         print(f"Error fetching {fips} {quarter}: {e}")
#         return None

In [24]:
,

''

In [ ]:
# def fetch_weather_full_county_6month(row):
#     lat = row.latitude
#     lon = row.longitude
#     fips = row.fips

#     county_weather = []

#     for year in years:
#         start_date = datetime.date(year, 1, 1)
#         end_date = datetime.date(year, 12, 31)

#         # adjust first & last year boundaries
#         if year == full_start_date.year:
#             start_date = full_start_date
#         if year == full_end_date.year:
#             end_date = full_end_date

#         try:
#             url = (
#                 f"https://archive-api.open-meteo.com/v1/archive"
#                 f"?latitude={lat}&longitude={lon}"
#                 f"&start_date={start_date}&end_date={end_date}"
#                 f"&daily=temperature_2m_max,temperature_2m_min,"
#                 f"precipitation_sum,snow_depth,windspeed_10m_max"
#                 f"&timezone=America/New_York"
#             )

#             response = requests.get(url, timeout=30)
#             data = response.json()

#             if "daily" in data:
#                 df_year = pd.DataFrame(data['daily'])
#                 df_year['date'] = pd.to_datetime(df_year['time'])
#                 df_year['fips'] = fips

#                 # --- Aggregate into 6-month periods ---
#                 df_year['year_half'] = df_year['date'].dt.year.astype(str) + '-' + \
#                                        ((df_year['date'].dt.month - 1) // 6 + 1).astype(str)

#                 agg_df = df_year.groupby(['fips', 'year_half']).agg({
#                     'temperature_2m_max':'mean',
#                     'temperature_2m_min':'mean',
#                     'precipitation_sum':'sum',
#                     'snow_depth':'sum',
#                     'windspeed_10m_max':'mean'
#                 }).reset_index()

#                 county_weather.append(agg_df)

#         except Exception as e:
#             print(f"Error {fips} year {year}: {e}")

#     if county_weather:
#         return pd.concat(county_weather, ignore_index=True)
#     else:
#         return None

In [ ]:
# from concurrent.futures import ThreadPoolExecutor

# weather_list = []

# with ThreadPoolExecutor(max_workers=10) as executor:
#     for idx, result in enumerate(
#         executor.map(fetch_weather_quarter, unique_county_quarters.itertuples())
#     ):
#         if result is not None:
#             weather_list.append(result)

#         if idx % 200 == 0:
#             print(f"Processed {idx} county-quarter pairs")

# weather_df = pd.concat(weather_list, ignore_index=True)

# print("Weather data shape:", weather_df.shape)

In [ ]:
# weather_list = []

# with ThreadPoolExecutor(max_workers=4) as executor:
#     for idx, result in enumerate(
#         executor.map(fetch_weather_full_county_6month, unique_counties.itertuples())
#     ):
#         if result is not None:
#             weather_list.append(result)

#         if idx % 50 == 0:
#             print(f"Processed {idx} counties")

# # --- Combine all counties ---
# if weather_list:
#     weather_df = pd.concat(weather_list, ignore_index=True)
#     print("6-Month aggregated weather shape:", weather_df.shape)
# else:
#     print("No weather data returned.")

In [ ]:
# weather_df['year_quarter'] = weather_df['date'].dt.to_period('Q')

In [ ]:
# quarterly_weather = (
#     weather_df
#     .groupby(['fips', 'year_quarter'])
#     .agg({
#         'temperature_2m_max': 'max',
#         'temperature_2m_min': 'min',
#         'precipitation_sum': 'sum',
#         'snow_depth': 'max',
#         'windspeed_10m_max': 'max'
#     })
#     .reset_index()
# )

# print("Quarterly shape:", quarterly_weather.shape)

In [ ]:
# weather_df.to_csv('/content/drive/MyDrive/full_daily_weather.csv', index=False)

In [ ]:
# weather_list = []

# for idx, row in unique_counties.iterrows():

#     lat = row['latitude_x']
#     lon = row['longitude_x']
#     fips = row['fips']

#     weather = fetch_weather(lat, lon, start_date, end_date)

#     if weather is not None:
#         weather['fips'] = fips
#         weather_list.append(weather)

#     if idx % 100 == 0:
#         print(f"Processed {idx} counties")

# weather_df = pd.concat(weather_list, ignore_index=True)

# print("Weather data shape:", weather_df.shape)

In [ ]:
# from concurrent.futures import ThreadPoolExecutor

# def fetch_weather_row(row):
#     lat, lon, fips, date = row.latitude, row.longitude, row.fips, row.date
#     weather = fetch_weather(lat, lon, date, date)
#     if weather is not None:
#         weather['fips'] = fips
#         weather['date'] = date
#     return weather

# weather_list = []

# with ThreadPoolExecutor(max_workers=20) as executor:
#     for idx, result in enumerate(executor.map(fetch_weather_row, unique_county_dates.itertuples())):
#         if result is not None:
#             weather_list.append(result)
#         if idx % 500 == 0:
#             print(f"Processed {idx} unique county-date pairs")

# weather_df = pd.concat(weather_list, ignore_index=True)

In [ ]:
# weather_df.to_csv("weather_data.csv", index=False)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [35]:
from datetime import datetime
from meteostat import Point, Daily

# Add columns for weather
df['temperature'] = None
df['wind_speed'] = None
df['wind_gust'] = None
df['rainfall'] = None
df['snow_depth'] = None
df['storm_flag'] = None

for i, row in df.iterrows():
    lat, lon = row['latitude'], row['longitude']
    date = pd.to_datetime(row['date'])

    if pd.isna(lat) or pd.isna(lon):
        continue

    location = Point(lat, lon)
    weather = Daily(location, date, date).fetch()

    if not weather.empty:
        df.at[i, 'temperature'] = weather['tavg'].values[0]
        df.at[i, 'wind_speed'] = weather['wspd'].values[0]
        df.at[i, 'wind_gust'] = weather['wpgt'].values[0]
        df.at[i, 'rainfall'] = weather['prcp'].values[0]
        df.at[i, 'snow_depth'] = weather['snow'].values[0]
        df.at[i, 'storm_flag'] = 1 if weather['wspd'].values[0] > 30 else 0

ImportError: cannot import name 'Daily' from 'meteostat' (/usr/local/lib/python3.12/dist-packages/meteostat/__init__.py)

In [ ]:
# Check missing values
print(df.isna().sum())



---



In [ ]:
import pandas as pd

# define column widths based on GHCN README
colspecs = [
    (0, 11),    # ID
    (12, 20),   # LATITUDE
    (21, 30),   # LONGITUDE
    (31, 37),   # ELEVATION
    (38, 40),   # STATE
    (41, 71),   # NAME
    (72, 75),   # GSN FLAG
    (76, 79),   # HCN/CRN FLAG
    (80, 85)    # WMO ID
]

columns = ['ID','LATITUDE','LONGITUDE','ELEVATION','STATE','NAME','GSN_FLAG','HCN_FLAG','WMO_ID']

stations = pd.read_fwf('ghcnd-stations.txt', colspecs=colspecs, names=columns)

stations = stations[['ID','LATITUDE','LONGITUDE','STATE','NAME']]
print("Stations loaded:", stations.shape)
stations.head()

In [ ]:
# import pandas as pd
# import numpy as np

# stations = pd.read_csv('ghcnd-stations.csv')  # adjust path
# stations = stations[['ID', 'LATITUDE', 'LONGITUDE', 'STATE', 'NAME']]

# print("Stations loaded:", stations.shape)
# stations.head()

In [ ]:
from scipy.spatial import cKDTree

# build KD-tree for fast nearest-neighbor search
station_coords = stations[['LATITUDE','LONGITUDE']].to_numpy()
tree = cKDTree(station_coords)

county_coords = unique_counties[['latitude','longitude']].to_numpy()
distances, indices = tree.query(county_coords, k=1)  # nearest station

# create mapping
unique_counties['station_id'] = stations.iloc[indices]['ID'].values

In [1]:
import pandas as pd

# get list of relevant station IDs
county_stations = unique_counties['station_id'].tolist()
weather_data_chunks = []

for year in range(full_start_date.year, full_end_date.year + 1):
    # Adjust the path to where your CSVs actually are
    file_path = f'/content/{year}.csv.gz'  # <-- directly in /content/

    try:
        for chunk in pd.read_csv(
            file_path,
            chunksize=2_000_000,   # safe for 12GB RAM
            compression='gzip',
            header=None,
            names=['ID','DATE','ELEMENT','VALUE','MFLAG','QFLAG','SFLAG','OBS_TIME']
        ):
            # filter only county stations and valid measurements
            chunk = chunk[chunk['ID'].isin(county_stations) & chunk['QFLAG'].isna()]

            if not chunk.empty:
                # pivot so each ELEMENT becomes a column
                chunk_pivot = chunk.pivot_table(
                    index=['ID', 'DATE'],
                    columns='ELEMENT',
                    values='VALUE'
                ).reset_index()

                weather_data_chunks.append(chunk_pivot)

    except FileNotFoundError:
        print(f"File for year {year} not found, skipping...")

# combine all chunks at the end
weather_df = pd.concat(weather_data_chunks, ignore_index=True)

# convert date
weather_df['DATE'] = pd.to_datetime(weather_df['DATE'], format='%Y%m%d')

print("Weather data shape:", weather_df.shape)
weather_df.head()

NameError: name 'unique_counties' is not defined

In [ ]:
# make sure the column names match
weather_df = weather_df.rename(columns={'ID': 'station_id'})

# merge with county info
weather_df = weather_df.merge(
    unique_counties[['fips', 'state', 'county', 'station_id']],
    on='station_id',
    how='left'
)

print(weather_df.head())

In [ ]:
weather_df.columns

In [ ]:
weather_df.to_csv('/content/drive/MyDrive/full_daily_weather.csv', index=False)



---



In [ ]:
weather_df = pd.read_csv('/content/drive/MyDrive/full_daily_weather.csv')

In [ ]:
# columns you want
keep_cols = [
    'station_id', 'DATE', 'fips',
    'TMAX', 'TMIN', 'TAVG',
    'PRCP', 'SNOW', 'SNWD',
    'AWND',
    'RHAV', 'RHMX', 'RHMN',
    'WT01', 'WT16', 'state', 'county'
]

weather_df = weather_df[keep_cols]

In [ ]:
# temperature conversion
for col in ['TMAX', 'TMIN', 'TAVG']:
    weather_df[col] = weather_df[col] / 10

# precipitation conversion
for col in ['PRCP', 'SNOW']:
    weather_df[col] = weather_df[col] / 10

In [ ]:
weather_df.isna().sum().sort_values(ascending=False)

In [ ]:
missing_percent = (
    weather_df.isna().mean() * 100
).sort_values(ascending=False)

missing_percent

In [ ]:
# keep only useful columns
weather_df = weather_df[
    ['station_id', 'DATE', 'fips',
     'TMAX', 'TMIN', 'PRCP', 'SNOW', 'state', 'county']
]

In [ ]:
weather_df[['PRCP','SNOW']] = weather_df[['PRCP','SNOW']].fillna(0)

In [ ]:
weather_df = weather_df.sort_values(['fips','DATE'])

weather_df[['TMAX','TMIN']] = (
    weather_df.groupby('fips')[['TMAX','TMIN']]
    .transform(lambda x: x.ffill())
)

In [ ]:
weather_df[['TMAX','TMIN']] = weather_df[['TMAX','TMIN']].fillna(
    weather_df[['TMAX','TMIN']].mean()
)

In [ ]:
missing_percent = (
    weather_df.isna().mean() * 100
).sort_values(ascending=False)

missing_percent

In [ ]:
weather_df.duplicated(subset=['fips','DATE']).sum()

In [ ]:
weather_df = weather_df.drop_duplicates(subset=['fips','DATE'])

In [ ]:
weather_df.dtypes

In [ ]:
weather_df['DATE'].min(), weather_df['DATE'].max()

In [ ]:
weather_df.describe()

In [ ]:
weather_df.columns

In [ ]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier

# # Only numerical weather features
# features = ['TMAX','TMIN','PRCP','SNOW']

# X = df[features].fillna(0)  # handle missing values
# y = np.random.randint(0, 2, size=len(df))  # pseudo-target

# model = XGBClassifier(n_estimators=200, random_state=42)
# model.fit(X, y)

# importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
# print("Feature importance:\n", importance)

In [ ]:
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# X = df[features].fillna(0)
# X_scaled = StandardScaler().fit_transform(X)

# pca = PCA(n_components=2)
# pca.fit(X_scaled)

# loadings = pd.DataFrame(pca.components_.T, index=features, columns=['PC1','PC2'])
# print("PCA loadings:\n", loadings)

In [ ]:
# variances = df[features].var().sort_values(ascending=False)
# print("Feature variance:\n", variances)

In [ ]:
poles_df = pd.read_csv('Pole_Attachments.csv')

In [ ]:
poles_df.columns

In [ ]:
import pandas as pd
from scipy.spatial import cKDTree

# Load county centroid data
counties = pd.read_csv('County Centroids.csv')  # must have fips, latitude, longitude

# Build KD-tree
county_coords = counties[['latitude','longitude']].to_numpy()
tree = cKDTree(county_coords)

# Pole coordinates
pole_coords = poles_df[['Location Latitude','Location Longitude']].to_numpy()
distances, indices = tree.query(pole_coords, k=1)

# Assign nearest county fips to poles
poles_df['fips'] = counties.iloc[indices]['fips'].values

In [ ]:
import numpy as np

# Convert lat/lon to float
poles_df['Location Latitude'] = pd.to_numeric(poles_df['Location Latitude'], errors='coerce')
poles_df['Location Longitude'] = pd.to_numeric(poles_df['Location Longitude'], errors='coerce')

# Drop rows with missing lat/lon
poles_clean = poles_df.dropna(subset=['Location Latitude', 'Location Longitude']).copy()

# Build KD-tree from county centroids
county_coords = counties[['latitude','longitude']].to_numpy()
tree = cKDTree(county_coords)

# Pole coordinates
pole_coords = poles_clean[['Location Latitude','Location Longitude']].to_numpy()

# Query nearest county
distances, indices = tree.query(pole_coords, k=1)

# Assign nearest fips
poles_clean['fips'] = counties.iloc[indices]['cfips'].values

print(poles_clean[['Pole Attachment ID', 'fips']].head())

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# Example: poles_df already loaded

# Extract latitude & longitude from 'Location' column
def extract_coords(location):
    try:
        if pd.isna(location):
            return np.nan, np.nan
        # Expect format "POINT (lon lat)"
        loc_str = location.strip().replace("POINT (", "").replace(")", "")
        lon, lat = map(float, loc_str.split())
        return lat, lon
    except:
        return np.nan, np.nan

poles_df['Latitude'], poles_df['Longitude'] = zip(*poles_df['Location'].map(extract_coords))

# Drop rows where coordinates are missing
poles_clean = poles_df.dropna(subset=['Latitude','Longitude']).copy()

print("Clean poles sample:")
print(poles_clean[['Pole Attachment ID', 'Latitude','Longitude']].head())

# Now you can build KD-tree and attach nearest county
county_coords = counties[['latitude','longitude']].to_numpy()
tree = cKDTree(county_coords)

pole_coords = poles_clean[['Latitude','Longitude']].to_numpy()
distances, indices = tree.query(pole_coords, k=1)

poles_clean['fips'] = counties.iloc[indices]['cfips'].values

print(poles_clean[['Pole Attachment ID', 'fips']].head(50))

In [ ]:
print(poles_clean[['Pole Attachment ID', 'fips']].head(25))

In [ ]:
# Convert Install Date to datetime, remove tz info
poles_clean['Install Date'] = pd.to_datetime(poles_clean['Install Date'], errors='coerce').dt.tz_localize(None)

# Drop rows where Install Date is NaT
poles_clean = poles_clean.dropna(subset=['Install Date'])

# Ensure weather_end is tz-naive
weather_end = pd.to_datetime(weather_df['DATE'].max()).tz_localize(None)

# Compute pole age in years
poles_clean['pole_age_years'] = (weather_end - poles_clean['Install Date']).dt.days / 365.25

poles_clean[['Install Date', 'pole_age_years']].head()

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import geopandas as gpd

# Ensure coordinates are floats
gdf_poles['Location Longitude'] = pd.to_numeric(gdf_poles['Location Longitude'], errors='coerce')
gdf_poles['Location Latitude'] = pd.to_numeric(gdf_poles['Location Latitude'], errors='coerce')
gdf_county['longitude'] = pd.to_numeric(gdf_county['longitude'], errors='coerce')
gdf_county['latitude'] = pd.to_numeric(gdf_county['latitude'], errors='coerce')

# Drop any rows with NaN coords
gdf_poles = gdf_poles.dropna(subset=['Location Longitude','Location Latitude'])
gdf_county = gdf_county.dropna(subset=['longitude','latitude'])

# Make 2D arrays for KD-tree
pole_points = gdf_poles[['Location Longitude','Location Latitude']].to_numpy()
county_points = gdf_county[['longitude','latitude']].to_numpy()

# Confirm shapes
print("Pole points shape:", pole_points.shape)
print("County points shape:", county_points.shape)

# Build KD-tree
tree = cKDTree(pole_points)

# Query nearest pole for each county
distances, indices = tree.query(county_points, k=1)

# Attach nearest pole info
gdf_county['nearest_pole_id'] = gdf_poles.iloc[indices]['Pole Attachment ID'].values
gdf_county['pole_age_years'] = gdf_poles.iloc[indices]['pole_age_years'].values

# Merge with weather_df on 'fips'
weather_df = weather_df.merge(
    gdf_county[['fips','nearest_pole_id','pole_age_years']],
    on='fips',
    how='left'
)

weather_df.head()



---



In [1]:
pip install meteostat

In [ ]:
!pip install --upgrade meteostat

In [4]:
import pandas as pd

# Load CSV(/content/odin-real-time-outages-county (3).csv)
df = pd.read_csv("odin-real-time-outages-county (3).csv")

In [5]:
df['geo_point_2d'].head(10)

,geo_point_2d
0,"33.79996451155127, -110.81209658207892"
1,"33.349092045960525, -112.49140181452469"
2,"30.440372849857212, -90.72784044258735"
3,"42.77461113908742, -92.31804391066815"
4,"37.78117274040125, -96.83907011299543"
5,"37.68470257220585, -97.46100886512676"
6,"41.88804749818017, -74.25850985743743"
7,"41.402030531404, -74.30578097513283"
8,"42.27656398597118, -74.12295513082604"
9,"35.475098918780766, -79.1714912683053"


In [6]:
print(df.columns)

Index(['cause', 'communityDescriptor', 'metersAffected', 'reportedStartTime',
       'statusKind', 'utilityDisclaimer', 'Incident', 'customersRestored',
       'EstimatedRestorationTime', 'causeKind', 'incident_cause',
       'incident_location', 'incident_location_kind', 'name', 'utility_id',
       'county', 'state', 'geom', 'geo_point_2d', 'centroid'],
      dtype='object')


In [7]:
# 1. Clean column names
df.columns = df.columns.str.strip()

# 2. Convert datetime
df['reportedStartTime'] = pd.to_datetime(df['reportedStartTime'], utc=True, format='ISO8601', errors='coerce')
df['date'] = df['reportedStartTime'].dt.date

# 3. Extract coordinates
coords = df['geo_point_2d'].str.split(',', expand=True)

df['latitude'] = pd.to_numeric(coords[0].str.strip(), errors='coerce')
df['longitude'] = pd.to_numeric(coords[1].str.strip(), errors='coerce')

# 4. Confirm
print(df[['latitude','longitude']].head())
print(df.columns)

    latitude   longitude
0  33.799965 -110.812097
1  33.349092 -112.491402
2  30.440373  -90.727840
3  42.774611  -92.318044
4  37.781173  -96.839070
Index(['cause', 'communityDescriptor', 'metersAffected', 'reportedStartTime',
       'statusKind', 'utilityDisclaimer', 'Incident', 'customersRestored',
       'EstimatedRestorationTime', 'causeKind', 'incident_cause',
       'incident_location', 'incident_location_kind', 'name', 'utility_id',
       'county', 'state', 'geom', 'geo_point_2d', 'centroid', 'date',
       'latitude', 'longitude'],
      dtype='object')


In [8]:
len(df)

2308

In [9]:
print(df['latitude'].dtype)
print(df['longitude'].dtype)
print(len(df))

float64
float64
2308


In [10]:
df = df.dropna(subset=['latitude', 'longitude', 'date'])

In [11]:
unique_locations = df[['latitude','longitude']].drop_duplicates().reset_index(drop=True)

In [13]:
stations = pd.read_fwf(
    "/content/ghcnd-stations.txt",
    colspecs=[
        (0, 11),   # ID
        (12, 20),  # LATITUDE
        (21, 30),  # LONGITUDE
        (31, 37),  # ELEVATION
        (38, 40),  # STATE
        (41, 71),  # NAME
        (72, 75),  # GSN FLAG
        (76, 79),  # HCN/CRN FLAG
        (80, 85)   # WMO ID
    ],
    names=[
        "ID", "LATITUDE", "LONGITUDE", "ELEVATION",
        "STATE", "NAME", "GSN_FLAG", "HCN_CRN_FLAG", "WMO_ID"
    ]
)

print(stations.head())

            ID  LATITUDE  LONGITUDE  ELEVATION STATE                   NAME  \
0  ACW00011604   17.1167   -61.7833       10.1   NaN  ST JOHNS COOLIDGE FLD   
1  ACW00011647   17.1333   -61.7833       19.2   NaN               ST JOHNS   
2  AE000041196   25.3330    55.5170       34.0   NaN    SHARJAH INTER. AIRP   
3  AEM00041194   25.2550    55.3640       10.4   NaN             DUBAI INTL   
4  AEM00041217   24.4330    54.6510       26.8   NaN         ABU DHABI INTL   

  GSN_FLAG HCN_CRN_FLAG   WMO_ID  
0      NaN          NaN      NaN  
1      NaN          NaN      NaN  
2      GSN          NaN  41196.0  
3      NaN          NaN  41194.0  
4      NaN          NaN  41217.0  


In [14]:
from scipy.spatial import cKDTree

station_coords = stations[['LATITUDE','LONGITUDE']].to_numpy()
tree = cKDTree(station_coords)

location_coords = unique_locations[['latitude','longitude']].to_numpy()
distances, indices = tree.query(location_coords, k=1)

unique_locations['station_id'] = stations.iloc[indices]['ID'].values

In [15]:
df = df.merge(unique_locations, on=['latitude','longitude'], how='left')

In [17]:
county_stations = df['station_id'].unique().tolist()

In [20]:
full_start_date = pd.to_datetime(df['date']).min()
full_end_date   = pd.to_datetime(df['date']).max()

print(full_start_date, full_end_date)

2026-02-03 00:00:00 2026-03-02 00:00:00


In [24]:
import pandas as pd

# get list of relevant station IDs
#county_stations = unique_counties['station_id'].tolist()
weather_data_chunks = []

for year in range(full_start_date.year, full_end_date.year + 1):
    # Adjust the path to where your CSVs actually are
    file_path = f'/content/{year}.csv.gz'  # <-- directly in /content/

    try:
        for chunk in pd.read_csv(
            file_path,
            chunksize=2_000_000,   # safe for 12GB RAM
            compression='gzip',
            header=None,
            names=['ID','DATE','ELEMENT','VALUE','MFLAG','QFLAG','SFLAG','OBS_TIME']
        ):
            # filter only county stations and valid measurements
            #chunk = chunk[chunk['ID'].isin(county_stations) & chunk['QFLAG'].isna()]
            required_elements = ['SNWD','TMIN','TMAX','SNOW','PRCP']

            chunk = chunk[
            chunk['ID'].isin(df['station_id']) &
            chunk['ELEMENT'].isin(required_elements) &
            chunk['QFLAG'].isna()
            ]

            if not chunk.empty:
                # pivot so each ELEMENT becomes a column
                chunk_pivot = chunk.pivot_table(
                    index=['ID', 'DATE'],
                    columns='ELEMENT',
                    values='VALUE'
                ).reset_index()

                weather_data_chunks.append(chunk_pivot)

    except FileNotFoundError:
        print(f"File for year {year} not found, skipping...")

# combine all chunks at the end
if weather_data_chunks:
    weather_df = pd.concat(weather_data_chunks, ignore_index=True)
else:
    print("No weather data found for available years.")

# convert date
weather_df['DATE'] = pd.to_datetime(weather_df['DATE'], format='%Y%m%d')

print("Weather data shape:", weather_df.shape)
weather_df.head()

Weather data shape: (2424, 7)


ELEMENT,ID,DATE,PRCP,SNOW,SNWD,TMAX,TMIN
0,US1AZYV0133,2026-01-01,84.0,NaN,NaN,NaN,NaN
1,US1AZYV0133,2026-01-02,53.0,NaN,NaN,NaN,NaN
2,US1AZYV0133,2026-01-03,0.0,NaN,NaN,NaN,NaN
3,US1AZYV0133,2026-01-04,0.0,0.0,NaN,NaN,NaN
4,US1AZYV0133,2026-01-05,0.0,0.0,NaN,NaN,NaN


In [29]:
weather_df = weather_df.rename(columns={'ID': 'station_id'})
weather_df['DATE'] = pd.to_datetime(weather_df['DATE'])

df['date'] = pd.to_datetime(df['date'])

df = df.merge(
    weather_df,
    left_on=['station_id','date'],
    right_on=['station_id','DATE'],
    how='left'
)

In [30]:
df.to_csv('/content/drive/MyDrive/test_weather_df.csv', index=False)

In [61]:
df = pd.read_csv('/content/drive/MyDrive/test_weather_df.csv')

In [62]:
df.columns

Index(['cause', 'communityDescriptor', 'metersAffected', 'reportedStartTime',
       'statusKind', 'utilityDisclaimer', 'Incident', 'customersRestored',
       'EstimatedRestorationTime', 'causeKind', 'incident_cause',
       'incident_location', 'incident_location_kind', 'name', 'utility_id',
       'county', 'state', 'geom', 'geo_point_2d', 'centroid', 'date',
       'latitude', 'longitude', 'station_id', 'DATE', 'PRCP', 'SNOW', 'SNWD',
       'TMAX', 'TMIN'],
      dtype='object')

In [54]:
poles_df = pd.read_csv('Pole_Attachments.csv')

In [55]:
print(poles_df.columns.tolist())

['Pole Attachment ID', 'Signal Name', 'Location Latitude', 'Location Longitude', 'Pole Attachment Description', 'Pole Attachment Provider', 'Created Date', 'Modified Date', 'Intersection Quadrant', 'Install Date', 'Status', 'Signal', 'ATD Location ID', 'Infrastructure Owner', 'Infrastructure Type', 'Provider ID', 'Location']


In [63]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# =====================================================
# 1️⃣ Extract Latitude/Longitude from poles_df['Location']
# =====================================================

def extract_lat_lon(point_str):
    if pd.isna(point_str) or str(point_str).strip() == '':
        return np.nan, np.nan

    # Remove 'POINT (' and ')', then split
    coords = point_str.replace("POINT (", "").replace(")", "").split()

    try:
        lon = float(coords[0])
        lat = float(coords[1])
        return lat, lon
    except:
        return np.nan, np.nan

poles_df[['Latitude','Longitude']] = poles_df['Location'].apply(
    lambda x: pd.Series(extract_lat_lon(x))
)

# Drop rows with missing coordinates
poles_df_clean = poles_df.dropna(subset=['Latitude','Longitude']).reset_index(drop=True)

# =====================================================
# 2️⃣ Clean weather_df coordinates (already done from your dataset)
# =====================================================

df = df.dropna(subset=['latitude','longitude']).reset_index(drop=True)

# Ensure numeric
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Optional: filter valid ranges
df = df[
    df['latitude'].between(-90, 90) &
    df['longitude'].between(-180, 180)
]

# =====================================================
# 3️⃣ Build KDTree for nearest-neighbor search
# =====================================================

pole_coords = poles_df_clean[['Latitude','Longitude']].values
tree = cKDTree(pole_coords)

weather_coords = df[['latitude','longitude']].values

# =====================================================
# 4️⃣ Find nearest pole for each weather point
# =====================================================

distances, indices = tree.query(weather_coords, k=1)

# =====================================================
# 5️⃣ Attach nearest pole info
# =====================================================

nearest_poles = poles_df_clean.iloc[indices].reset_index(drop=True)
nearest_poles['Distance_to_Weather'] = distances

# =====================================================
# 6️⃣ Merge results
# =====================================================

merged_df = pd.concat(
    [df.reset_index(drop=True),
     nearest_poles.add_prefix('Pole_')],
    axis=1
)

print(merged_df.head())

                   cause  communityDescriptor  metersAffected  \
0                    NaN                 4007              46   
1                    NaN                 4013              34   
2  Pending Investigation                19017               1   
3                    NaN                36111               3   
4                    NaN                36071               3   

           reportedStartTime              statusKind  \
0  2026-03-02 15:10:00+00:00                     NaN   
1  2026-03-02 15:00:00+00:00                     NaN   
2  2026-03-02 08:52:00+00:00  awaitingCrewAssignment   
3  2026-03-02 14:34:29+00:00                 arrived   
4  2026-03-02 14:58:51+00:00                 arrived   

                                   utilityDisclaimer  \
0                                                NaN   
1                                                NaN   
2               Butler County Rural Elec Coop - (IA)   
3  Based on reports from customers and field per

In [65]:
merged_df.shape

(1629, 50)

In [66]:
merged_df.columns

Index(['cause', 'communityDescriptor', 'metersAffected', 'reportedStartTime',
       'statusKind', 'utilityDisclaimer', 'Incident', 'customersRestored',
       'EstimatedRestorationTime', 'causeKind', 'incident_cause',
       'incident_location', 'incident_location_kind', 'name', 'utility_id',
       'county', 'state', 'geom', 'geo_point_2d', 'centroid', 'date',
       'latitude', 'longitude', 'station_id', 'DATE', 'PRCP', 'SNOW', 'SNWD',
       'TMAX', 'TMIN', 'Pole_Pole Attachment ID', 'Pole_Signal Name',
       'Pole_Location Latitude', 'Pole_Location Longitude',
       'Pole_Pole Attachment Description', 'Pole_Pole Attachment Provider',
       'Pole_Created Date', 'Pole_Modified Date', 'Pole_Intersection Quadrant',
       'Pole_Install Date', 'Pole_Status', 'Pole_Signal',
       'Pole_ATD Location ID', 'Pole_Infrastructure Owner',
       'Pole_Infrastructure Type', 'Pole_Provider ID', 'Pole_Location',
       'Pole_Latitude', 'Pole_Longitude', 'Pole_Distance_to_Weather'],
      dtyp

In [69]:
print("Total Pole IDs:", merged_df["Pole_Pole Attachment ID"].nunique())

Total Pole IDs: 6


In [70]:
pole_id_column = "Pole_Pole Attachment ID"

print("Total rows:", len(merged_df))
print("Missing IDs:", merged_df[pole_id_column].isna().sum())

Total rows: 1629
Missing IDs: 0


In [71]:
duplicate_count = merged_df[pole_id_column].duplicated().sum()
print("Duplicate IDs:", duplicate_count)

Duplicate IDs: 1623


In [74]:
# Keep only one record per unique Pole ID
unique_poles_df = merged_df.drop_duplicates(subset=['Pole_Pole Attachment ID']).reset_index(drop=True)

print("Unique poles shape:", unique_poles_df.shape)
print(unique_poles_df[['Pole_Pole Attachment ID', 'latitude', 'longitude']])

Unique poles shape: (6, 50)
   Pole_Pole Attachment ID   latitude   longitude
0                     1212  33.799965 -110.812097
1                      592  33.349092 -112.491402
2                      738  42.774611  -92.318044
3                     1147  41.888047  -74.258510
4                      336  29.857466  -95.393015
5                      771  40.666849 -111.923821


In [72]:
# Count missing values per column
missing_counts = merged_df.isna().sum()

# Percentage of missing values per column
missing_percent = (missing_counts / len(merged_df)) * 100

# Combine into one table
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percent (%)': missing_percent
}).sort_values(by='Missing Percent (%)', ascending=False)

print(missing_summary)

                                  Missing Count  Missing Percent (%)
Pole_Signal Name                           1629           100.000000
Pole_Location Longitude                    1629           100.000000
Pole_Install Date                          1629           100.000000
Pole_Location Latitude                     1629           100.000000
utilityDisclaimer                          1575            96.685083
SNWD                                       1571            96.439533
SNOW                                       1558            95.641498
customersRestored                          1530            93.922652
TMAX                                       1502            92.203806
TMIN                                       1502            92.203806
PRCP                                       1491            91.528545
Pole_Signal                                1454            89.257213
DATE                                       1424            87.415592
causeKind                         

In [76]:
%cd /content/drive/MyDrive/ODIN.ipynb
!git init
!git remote add origin https://github.com/hathumiamarasinghe/pole-risk-modeling.git
!git add .
!git commit -m "Initial upload"
!git push -u origin main

[Errno 2] No such file or directory: '/content/drive/MyDrive/ODIN.ipynb'
/content
Reinitialized existing Git repository in /content/.git/
error: remote origin already exists.
^C
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@7e8dae2ee6c1.(none)')
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/hathumiamarasinghe/pole-risk-modeling.git'
